In [ ]:
import dash
from dash import html, dcc
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import textwrap

def create_timeline_app(df, resample_freq='D'):
    """
    Create a Dash app with scrollable cluster timeline.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing columns: 
        - date: datetime
        - interests: str
        - cluster_label: str
        - merged_cluster_label: int
        - cluster_category: str
    resample_freq : str, optional (default='W')
        Frequency to resample the data
    """
    # Convert date to datetime if not already
    df['date'] = pd.to_datetime(df['date'])
    
    def format_hover_text(date, interests, category, cluster, merged_label):
        # Break long interests into multiple lines with a max width of 40 characters
        if isinstance(interests, str):
            wrapped_interests = '<br>'.join(textwrap.wrap(interests, width=40))
        else:
            wrapped_interests = interests

        if isinstance(category, str):
            wrapped_category = '<br>'.join(textwrap.wrap(category, width=40))
        else:
            wrapped_category = category
            
        return (f"<b>Date:</b> {date:%Y-%m-%d}<br>"
                f"<b>Cluster:</b> {cluster}<br>"
                f"<b>Group:</b> {merged_label}<br>"
                f"<b>Category:</b> <br>{wrapped_category}<br>"
                f"<b>Interests:</b><br>{wrapped_interests}")
    
    # Create the main timeline figure
    fig = go.Figure()
    
    # Get unique clusters and sort them by merged_cluster_label and cluster_label
    cluster_info = (df.groupby(['cluster_label'])
                   .agg({'merged_cluster_label': 'first'})
                   .reset_index())
    cluster_info = cluster_info.sort_values(['merged_cluster_label', 'cluster_label'])
    clusters = cluster_info['cluster_label'].tolist()
    
    # Create color map for merged labels (using integers as keys)
    unique_merged_labels = sorted(df['merged_cluster_label'].unique())
    colors = {
        label: f'hsl({(i * 360) // len(unique_merged_labels)}, 70%, 70%)'
        for i, label in enumerate(unique_merged_labels)
    }
    
    # Calculate total height (15 pixels per cluster)
    spacing_per_cluster = 15
    total_height = len(clusters) * spacing_per_cluster
    
    # Process each cluster
    for i, cluster in enumerate(clusters):
        cluster_data = df[df['cluster_label'] == cluster].copy()
        
        # Get the merged label and category for this cluster
        merged_label = cluster_data['merged_cluster_label'].iloc[0]
        category = cluster_data['cluster_category'].iloc[0]
        
        # Get start and end dates
        start_date = cluster_data['date'].min()
        end_date = cluster_data['date'].max()
        
        # Create resampled data for hover information
        if resample_freq != 'D':
            # Group by resampled date and collect all interests in that period
            cluster_data.set_index('date', inplace=True)
            resampled = cluster_data.resample(resample_freq)['interests'].agg(lambda x: '<br>'.join(textwrap.wrap('<br>'.join(x), width=40)))
            dates = resampled.index
            interests = resampled.values
        else:
            dates = cluster_data['date']
            interests = cluster_data['interests']
        
        # Add main line
        fig.add_trace(
            go.Scatter(
                x=[start_date, end_date],
                y=[i, i],
                mode='lines',
                line=dict(
                    color=colors[merged_label],
                    width=5
                ),
                name=f"{cluster} ({category})",
                showlegend=False
            )
        )
        
        # Add invisible scatter points for hover information
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=[i] * len(dates),
                mode='markers',
                marker=dict(
                      size=3, 
                      color="black",  
                      opacity=0.7  
                  ),
                name='',
                text=[format_hover_text(d, i, category, cluster, merged_label)
                      for d, i in zip(dates, interests)],
                hoverinfo='text',
                showlegend=False
            )
        )
    
    # Update layout
    fig.update_layout(
        title=None,
        xaxis=dict(
            title='Date',
            showgrid=True,
            gridwidth=1,
            gridcolor='LightGrey'
        ),
        yaxis=dict(
            title=None,
            showgrid=True,
            gridwidth=1,
            gridcolor='LightGrey',
            ticktext=clusters,
            tickvals=list(range(len(clusters))),
            autorange="reversed"
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_family="Arial",
            align="left"
        ),
        plot_bgcolor='white',
        showlegend=False,
        height=total_height + 50,
        margin=dict(t=10, l=100, r=50, b=50)
    )
    
    # Create a small figure for the header (optional)
    header_fig = go.Figure()
    header_fig.update_layout(
        height=50,
        margin=dict(t=0, l=100, r=50, b=0),
        xaxis=dict(
            title='Timeline',
            showgrid=False,
            zeroline=False,
            showticklabels=False
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=False
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Create the Dash app
    app = dash.Dash(__name__)
    
    # Define the layout
    app.layout = html.Div([
        # Header (optional)
        dcc.Graph(
            figure=header_fig,
            config={'displayModeBar': False}
        ),
        # Scrollable timeline
        html.Div([
            dcc.Graph(
                figure=fig,
                config={'displayModeBar': True}
            )
        ], style={'overflow-y': 'auto', 'height': '600px'}),
    ])
    
    return app

In [ ]:
import pandas as pd

df = pd.read_parquet("/Users/ma9o/Desktop/enclaveid/apps/data-pipeline/data/clusters_categories/cm0i27jdj0000aqpa73ghpcxf.snappy")

# Create and run the app
app = create_timeline_app(df, resample_freq='D')

app.run(debug=False)